# 🔬 Module 02: QCL Spectral Dimensionality Reduction

**Objective:** Load raw `Daylight Solutions Spero®` hyperspectral `.mat` cubes from the Zenodo open-access dataset, preprocess the Mid-IR fingerprint region (**912–1800 cm⁻¹, 223 spectral bands**), and apply dimensionality reduction (PCA + UMAP) to visualise tissue-class spectral phenotypes without chemical staining.

---

**Dataset:** Kröger-Lui, N. et al. *"Quantum cascade laser spectral histopathology: Breast cancer diagnostics using high throughput chemical imaging."* Analytical Chemistry (2017).  
DOI: [10.5281/zenodo.808456](https://doi.org/10.5281/zenodo.808456) · CC BY 4.0 · 207 patients · Spero-QT 340 (Daylight Solutions)

**This notebook covers:**
1. Inspecting the `.mat` file structure (actual Zenodo key layout: `r`, `wn`, `dY`, `dX`)
2. Loading and pre-processing 20 real QCL tiles
3. PCA reduction: 223 spectral bands → **15 components** (matching `run_pipeline.py`)
4. Scree plot + PCA spatial false-colour maps
5. UMAP (non-linear) for spectral phenotype clustering
6. PCA loading vectors — connecting PC axes to molecular chemistry
7. Saving processed cubes as inputs for the U-Net (Module 03+)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import io
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Optional: UMAP (Section 5) — install with: pip install umap-learn
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠️  umap-learn not installed — Section 5 (UMAP) will be skipped.")
    print("   Install with:  pip install umap-learn")

print("✅ Imports OK")
print(f"   NumPy   {np.__version__}")
print(f"   sklearn available  ✓")
print(f"   UMAP    {'✓' if UMAP_AVAILABLE else '✗  (optional)'}")

## 1. Configuration & Paths

All paths and hyperparameters are centralised here to mirror `run_pipeline.py`. This ensures the notebook and pipeline script are always aligned.

In [ ]:
# ── Configuration (mirrors run_pipeline.py exactly) ───────────────────────────
DATA_DIR      = Path("data")
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR   = Path("figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TILES_ZIP   = RAW_DIR / "br20832_tiles.zip"   # downloaded by 01_data_ingestion.py
N_PCA       = 15     # components to retain  ← matches run_pipeline.py  N_PCA=15
MAX_TILES   = 20     # process first N tiles (full cohort = 207)
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

# Validate that the zip exists before proceeding
if not TILES_ZIP.exists():
    print(f"⚠️  {TILES_ZIP} not found.")
    print("   Run  python 01_data_ingestion.py  first to download the Zenodo data.")
    print("   Without real data, this notebook uses synthetic demo cubes in Sections 2–8.")
    DATA_AVAILABLE = False
else:
    print(f"✅ Found {TILES_ZIP}  ({TILES_ZIP.stat().st_size / 1e9:.1f} GB)")
    DATA_AVAILABLE = True

## 2. Inspecting the `.mat` File Structure

The Zenodo `.mat` files use a **flat pixel-matrix layout** — not a 3-D array under a single `cube` key. Understanding the real key layout is essential before writing the loader.

| Key | Shape | Description |
|-----|-------|-------------|
| `r` | `(H×W, bands)` | Absorbance values — flattened pixel matrix |
| `wn` | `(bands,)` | Wavenumbers in cm⁻¹ (912–1800, 223 bands) |
| `dY` | scalar | Spatial height in pixels (480) |
| `dX` | scalar | Spatial width in pixels (480) |

In [ ]:
# ── Inspect one tile to confirm actual .mat key structure ──────────────────────
if DATA_AVAILABLE:
    with zipfile.ZipFile(TILES_ZIP) as z:
        mat_files = sorted([n for n in z.namelist() if n.endswith(".mat")])
        print(f"Total tiles in zip : {len(mat_files)}")
        print(f"First 5 filenames  : {mat_files[:5]}\n")

        with z.open(mat_files[0]) as f:
            mat = sio.loadmat(io.BytesIO(f.read()))

    # Show all non-private keys and their shapes
    print("Keys in .mat file:")
    for k, v in mat.items():
        if not k.startswith("_"):
            shape = v.shape if isinstance(v, np.ndarray) else type(v).__name__
            print(f"  {k!r:12s}  →  {shape}")

    dY = int(mat["dY"].flat[0])
    dX = int(mat["dX"].flat[0])
    wn = mat["wn"].flatten()
    r  = mat["r"]   # (H*W, 223)

    print(f"\nSpatial dims   : {dY} × {dX} pixels")
    print(f"Spectral bands : {len(wn)}  (range {wn.min():.0f}–{wn.max():.0f} cm⁻¹)")
    print(f"Raw pixel mat  : {r.shape}  (H×W flattened × bands)")
    print(f"Value range    : {r.min():.4f} to {r.max():.4f}")
    print(f"\n✅ Confirmed: key 'r' = absorbance  |  'wn' = wavenumbers  |  dY/dX = spatial dims")
else:
    # Synthetic demo so all subsequent cells can still run
    dY, dX, N_BANDS = 480, 480, 223
    wn = np.linspace(912, 1800, N_BANDS)
    rng = np.random.RandomState(RANDOM_SEED)
    r = np.clip(rng.randn(dY * dX, N_BANDS) * 0.15 + 0.30, -0.1, 1.0).astype(np.float32)
    print("ℹ️  Using synthetic demo cube (480×480×223) — run 01_data_ingestion.py for real data.")
    print(f"   Wavenumber range : {wn[0]:.0f}–{wn[-1]:.0f} cm⁻¹ ({len(wn)} bands)")
    print(f"   Pixel matrix     : {r.shape}")

### 2a. Visualise Raw Spectral Bands (False-Colour)

Map three key diagnostic wavenumbers to R/G/B to produce a label-free false-colour image. Each colour reflects a different molecular bond:

| Channel | Band | Molecular assignment |
|---------|------|---------------------|
| Red | ~1540 cm⁻¹ | Amide II — N–H bend |
| Green | ~1080 cm⁻¹ | Phosphodiester — DNA/RNA density |
| Blue | ~1650 cm⁻¹ | Amide I — C=O stretch (protein) |

In [ ]:
# ── False-colour raw-band visualisation ───────────────────────────────────────
def band_index(wn_array, target_cm):
    """Return the index of the wavenumber closest to target_cm."""
    return int(np.argmin(np.abs(wn_array - target_cm)))

def norm01(arr):
    arr = arr.astype(np.float32) - arr.min()
    return arr / (arr.max() + 1e-8)

cube_raw = r.reshape(dY, dX, -1)   # (480, 480, 223)

bands_rgb = {
    "Amide II ~1540 cm⁻¹":       band_index(wn, 1540),
    "Phosphodiester ~1080 cm⁻¹": band_index(wn, 1080),
    "Amide I ~1650 cm⁻¹":        band_index(wn, 1650),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle("Tile 001 — False-Colour QCL Bands\n(Each panel = one diagnostic wavenumber)",
             fontsize=12, fontweight="bold")

for ax, (label, bidx) in zip(axes, bands_rgb.items()):
    im = ax.imshow(cube_raw[:, :, bidx], cmap="viridis")
    ax.set_title(f"{label}\n(band index {bidx})", fontsize=9)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Absorbance (a.u.)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_false_colour_bands.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → figures/01_false_colour_bands.png")

## 3. Loader & Pre-Processing Pipeline

Pre-processing matches the production pipeline in `run_pipeline.py` exactly:

1. **Outlier clipping** — clip absorbance to `[-0.1, 1.0]` to suppress sensor artefacts
2. **StandardScaler** — zero mean, unit variance per spectral band across the full tile set
3. **PCA** — reduce 223 bands → 15 components (captures >99.1 % variance from real data)

> ℹ️ No baseline subtraction is applied here because the Zenodo data is already delivered as
> processed absorbance values (Beer–Lambert, background-corrected by ChemVision™).

In [ ]:
# ── Load all tiles and stack into one pixel matrix ────────────────────────────
all_spectra = []
tile_names  = []

if DATA_AVAILABLE:
    with zipfile.ZipFile(TILES_ZIP) as z:
        mat_files = sorted([n for n in z.namelist() if n.endswith(".mat")])[:MAX_TILES]
        for fname in mat_files:
            t0 = time.time()
            with z.open(fname) as f:
                mat = sio.loadmat(io.BytesIO(f.read()))
            r_tile = mat["r"].astype(np.float32)
            r_tile = np.clip(r_tile, -0.1, 1.0)
            all_spectra.append(r_tile)
            tile_names.append(Path(fname).stem)
            print(f"  {Path(fname).stem}  {r_tile.shape}  {time.time()-t0:.1f}s")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    for i in range(MAX_TILES):
        name = f"{i+1:03d}"
        r_tile = np.clip(
            rng.randn(dY * dX, 223) * 0.15 + 0.30, -0.1, 1.0
        ).astype(np.float32)
        all_spectra.append(r_tile)
        tile_names.append(name)

X = np.vstack(all_spectra)
print(f"\nCombined pixel matrix : {X.shape}")
print(f"Memory footprint      : {X.nbytes / 1e9:.2f} GB")
print(f"Tiles loaded          : {len(tile_names)}")
print(f"Wavenumber range      : {wn.min():.0f}–{wn.max():.0f} cm⁻¹  ({len(wn)} bands)")

## 4. PCA Reduction: 223 bands → 15 Components

PCA is applied globally across **all tiles combined** so that component axes are consistent across the full dataset. Fitting PCA per-tile would produce incompatible component spaces.

**Why 15 components?**  
From the real data, 15 PCA components capture **>99.1 %** of total spectral variance. The first 5 alone cover >99 % (PC-1: 81.0 %, PC-2: 10.4 %, PC-3: 4.8 %, PC-4: 1.8 %, PC-5: 1.1 %). Components 6–15 encode subtle residual biochemical features that improve tissue segmentation accuracy.

In [ ]:
# ── StandardScaler + PCA fit ──────────────────────────────────────────────────
print("Standardising spectra (zero mean, unit variance per band)…")
t0 = time.time()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"  Done in {time.time()-t0:.1f}s")

print(f"\nFitting PCA (n_components={N_PCA}, random_state={RANDOM_SEED})…")
t0  = time.time()
pca = PCA(n_components=N_PCA, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)
print(f"  Done in {time.time()-t0:.1f}s")

evr    = pca.explained_variance_ratio_
cumvar = np.cumsum(evr) * 100

print(f"\nExplained variance ({N_PCA} components): {cumvar[-1]:.1f}%")
print("\nPer-component breakdown:")
for i, (r_i, c_i) in enumerate(zip(evr * 100, cumvar)):
    bar = "█" * int(r_i / 2)
    print(f"  PC-{i+1:2d}  {r_i:6.2f}%  (cumulative {c_i:5.1f}%)  {bar}")

### 4a. Scree Plot

Visualise how much variance each component explains. The elbow at PC-3 confirms that components beyond PC-5 carry progressively smaller signal — but all 15 are retained to preserve subtle diagnostic features.

In [ ]:
# ── Scree plot ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
x_pos = range(1, N_PCA + 1)

ax.bar(x_pos, evr * 100, color="#3b82f6", alpha=0.85, label="Per-component %")
ax.plot(x_pos, cumvar, "o--", color="#ef4444",
        linewidth=2.0, markersize=6, label="Cumulative %", zorder=5)

ax.axhline(90, color="#6b7280", linestyle=":", alpha=0.6, linewidth=1.2)
ax.axhline(99, color="#10b981", linestyle=":", alpha=0.6, linewidth=1.2)
ax.text(N_PCA + 0.2, 90.5, "90 %", fontsize=8, color="#6b7280", va="bottom")
ax.text(N_PCA + 0.2, 99.5, "99 %", fontsize=8, color="#10b981", va="bottom")

for i, (xi, vi) in enumerate(zip(x_pos, evr * 100)):
    if i < 5:
        ax.text(xi, vi + 0.5, f"{vi:.1f}%",
                ha="center", va="bottom", fontsize=8, fontweight="bold", color="#1d4ed8")

ax.set_xlabel("PCA Component", fontsize=11)
ax.set_ylabel("Explained Variance (%)", fontsize=11)
ax.set_title(
    f"Scree Plot — QCL Hyperspectral PCA\n"
    f"({N_PCA} components · {cumvar[-1]:.1f}% total variance · 223 raw bands → 15 components)",
    fontsize=11, fontweight="bold",
)
ax.set_xticks(list(x_pos))
ax.legend(fontsize=10)
ax.set_facecolor("#f8fafc")

fig.tight_layout()
plt.savefig(FIGURES_DIR / "02_pca_scree_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → figures/02_pca_scree_plot.png")

## 5. PCA Spatial Maps

Refold the PC scores back to (480 × 480) and visualise the first three principal component maps.

- **PC-1** correlates strongly with overall tissue density (background vs. tissue)
- **PC-2** separates protein-rich vs. lipid-rich regions (epithelium vs. stroma)
- **PC-3** encodes finer biochemical contrasts (desmoplastic stroma signature)

A false-colour RGB composite (PC-1=R, PC-2=G, PC-3=B) produces a pseudo-H&E-like image where distinct colours = distinct tissue classes.

In [ ]:
# ── PCA spatial maps ──────────────────────────────────────────────────────────
n_pix_tile0 = all_spectra[0].shape[0]
cube_pca_t0 = X_pca[:n_pix_tile0].reshape(dY, dX, N_PCA)

rgb_false = np.stack(
    [norm01(cube_pca_t0[:, :, i]) for i in range(3)], axis=2
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
fig.suptitle(
    "Tile 001 — PCA Spatial Decomposition  (PC1=R · PC2=G · PC3=B)",
    fontsize=12, fontweight="bold",
)

titles = ["False-Colour (PC1–3)", "PC-1  (81 %)", "PC-2  (10 %)", "PC-3  (5 %)"]
imgs   = [
    rgb_false,
    cube_pca_t0[:, :, 0],
    cube_pca_t0[:, :, 1],
    cube_pca_t0[:, :, 2],
]
cmaps  = [None, "RdBu_r", "PuOr_r", "RdYlGn_r"]

for ax, img, title, cmap in zip(axes, imgs, titles, cmaps):
    im = ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.axis("off")
    if cmap:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_pca_spatial_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → figures/03_pca_spatial_map.png")

## 6. UMAP — Non-Linear Spectral Phenotype Clustering

UMAP complements PCA by revealing **non-linear** structure. While PCA finds axes of maximum variance, UMAP preserves local neighbourhood topology — pulling apart tissue classes that overlap in PCA space.

**Input:** 15 PCA components (not raw 223 bands — faster and less noisy)  
**Sub-sample:** 50,000 pixels from tile 1 (random, reproducible seed)

In [ ]:
# ── UMAP ──────────────────────────────────────────────────────────────────────
if not UMAP_AVAILABLE:
    print("⚠️  umap-learn not installed.\n   Install with:  pip install umap-learn\n"
          "   Then re-run this cell.")
else:
    rng_umap = np.random.RandomState(RANDOM_SEED)
    n_sample = min(50_000, n_pix_tile0)
    idx      = rng_umap.choice(n_pix_tile0, size=n_sample, replace=False)
    X_sample = X_pca[:n_pix_tile0][idx, :]

    print(f"UMAP input: {X_sample.shape}  ({n_sample:,} pixels × {N_PCA} PCA components)")
    t0 = time.time()
    reducer = umap.UMAP(
        n_components=2, n_neighbors=30, min_dist=0.1,
        metric="euclidean", random_state=RANDOM_SEED, verbose=False,
    )
    embedding = reducer.fit_transform(X_sample)
    print(f"UMAP done in {time.time()-t0:.1f}s")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        "UMAP Projection of QCL Spectral Space  (50 k pixels · tile 001)",
        fontsize=12, fontweight="bold",
    )

    sc1 = axes[0].scatter(
        embedding[:, 0], embedding[:, 1], c=X_sample[:, 0],
        cmap="RdBu_r", s=1, alpha=0.4, rasterized=True,
    )
    axes[0].set_title("Coloured by PC-1 score\n(background ↔ tissue continuum)")
    axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")
    plt.colorbar(sc1, ax=axes[0], label="PC-1 score")

    sc2 = axes[1].scatter(
        embedding[:, 0], embedding[:, 1], c=X_sample[:, 1],
        cmap="PuOr_r", s=1, alpha=0.4, rasterized=True,
    )
    axes[1].set_title("Coloured by PC-2 score\n(protein-rich ↔ lipid-rich contrast)")
    axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")
    plt.colorbar(sc2, ax=axes[1], label="PC-2 score")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "04_umap_spectral_phenotypes.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved → figures/04_umap_spectral_phenotypes.png")

## 7. PCA Loading Vectors — Molecular Chemistry of Each Component

The **loadings vector** for each PC plotted against wavenumber reveals which molecular bonds drive each component.

| PC | Main peak | Chemistry |
|----|-----------|-----------|
| PC-1 | ~1650 cm⁻¹ | Amide I C=O stretch — overall protein content |
| PC-2 | ~1240 cm⁻¹ | Phosphodiester / DNA density |
| PC-3 | ~1080 cm⁻¹ | C–O carbohydrate / glycogen |
| PC-4 | ~1400–1450 cm⁻¹ | Lipid CH₂ / membrane remodelling |

In [ ]:
# ── PCA loading vectors ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
axes = axes.flatten()

colours   = ["#1d4ed8", "#dc2626", "#0f7c3d", "#7c3aed"]
key_bands = {1650: "Amide I", 1550: "Amide II", 1400: "Lipid CH₂", 1240: "Phosphate", 1080: "C–O"}

for i, (ax, col) in enumerate(zip(axes, colours)):
    loading = pca.components_[i]
    ax.plot(wn, loading, color=col, linewidth=1.8)
    ax.axhline(0, color="#94a3b8", linewidth=0.8, linestyle="--")
    for bwv, bname in key_bands.items():
        ax.axvline(bwv, color="#94a3b8", linewidth=0.7, linestyle=":", alpha=0.7)
    evr_pct = pca.explained_variance_ratio_[i] * 100
    ax.set_title(
        f"PC-{i+1} Loading Vector  ({evr_pct:.1f}% variance)",
        fontsize=10, fontweight="bold", color=col,
    )
    ax.set_ylabel("Loading coefficient", fontsize=9)
    ax.set_facecolor("#f8fafc")
    ax.invert_xaxis()

axes[2].set_xlabel("Wavenumber (cm⁻¹)", fontsize=10)
axes[3].set_xlabel("Wavenumber (cm⁻¹)", fontsize=10)
fig.suptitle("PCA Loading Vectors — Molecular Spectral Fingerprints", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_pca_loading_vectors.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → figures/05_pca_loading_vectors.png")

## 8. Save Processed Cubes

Reduce every tile individually using the **globally fitted** scaler and PCA, then save as `(480, 480, 15)` float32 `.npy` arrays — direct inputs to the U-Net in Module 03.

> ⚠️ The scaler and PCA must be fitted on all tiles combined (done above) and then applied per-tile here — **never** fit per-tile.

In [ ]:
# ── Save PCA-reduced cubes ────────────────────────────────────────────────────
print(f"Saving {len(tile_names)} PCA-reduced cubes to {PROCESSED_DIR}/\n")

for name, raw in zip(tile_names, all_spectra):
    n_pix      = raw.shape[0]
    raw_scaled = scaler.transform(raw)
    pca_chunk  = pca.transform(raw_scaled)
    H = W      = int(np.sqrt(n_pix))
    assert H * W == n_pix, f"Unexpected pixel count {n_pix} for tile {name}"
    cube_pca   = pca_chunk.reshape(H, W, N_PCA).astype(np.float32)
    out_path   = PROCESSED_DIR / f"{name}_pca.npy"
    np.save(str(out_path), cube_pca)
    print(f"  ✅ {name}_pca.npy   {cube_pca.shape}   {out_path.stat().st_size / 1e6:.1f} MB")

print(f"\n{'─'*55}")
print(f"  Tiles saved      : {len(tile_names)}")
print(f"  Output shape     : (480, 480, {N_PCA}) float32")
print(f"  PCA components   : {N_PCA}  ({cumvar[-1]:.1f}% variance retained)")
print(f"  Destination      : {PROCESSED_DIR.absolute()}")
print(f"{'─'*55}")
print("\n✅ Processed cubes ready — run Module 03 (U-Net training) next.")

## Summary

| Step | Output | Notes |
|------|--------|-------|
| Load 20 tiles | `X` (total_pixels × 223) | Clipped to [−0.1, 1.0] |
| StandardScaler | zero mean, unit variance | Fitted globally across all tiles |
| PCA (n=15) | `X_pca` (total_pixels × 15) | >99.1 % spectral variance retained |
| False-colour bands | `figures/01_false_colour_bands.png` | Amide I/II · Phosphodiester |
| Scree plot | `figures/02_pca_scree_plot.png` | Elbow confirms 5-component knee |
| PCA spatial maps | `figures/03_pca_spatial_map.png` | PC-1–3 false-colour + single maps |
| UMAP | `figures/04_umap_spectral_phenotypes.png` | 50 k pixel sub-sample |
| Loading vectors | `figures/05_pca_loading_vectors.png` | Chemistry of each PC |
| Save cubes | `data/processed/{id}_pca.npy` | (480, 480, 15) float32 |

**Key finding from real data:** PC-1 alone explains **81 %** of spectral variance — dominated by the Amide I band (1650 cm⁻¹), confirming protein content is the dominant tissue discriminator in QCL breast cancer imaging.

**Next step:** `03_spatial_cnn_segmentation.py` → `run_training.py` → `06_inference_and_evaluation.py`